# Weights & Biases Demo w/ CNN + MNIST

You will learn:

- Hyperparameter logging: `wandb.config`
- Live loss / accuracy curves: `wandb.log`
- Saving model checkpoints: `wandb.artifact`


In [1]:
"""
Import libraries.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import wandb

print("Libraries imported.")


Libraries imported.


In [2]:
"""
CNN model class.
"""
class CNN(nn.Module):
    def __init__(self, conv1_filters, conv2_filters, fc_hidden):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, conv1_filters, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                          # 28x28 -> 14x14
            nn.Conv2d(conv1_filters, conv2_filters, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                          # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv2_filters * 7 * 7, fc_hidden),
            nn.ReLU(),
            nn.Linear(fc_hidden, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

print("Model class defined.")
        

Model class defined.


In [3]:
"""
Set up configurations.
"""

# Training configs
LR  = 1e-3   # learning rate
BS  = 64     # batch size
EPOCHS = 10   # epochs

# Model configs
C1F = 32     # conv1 filters
C2F = 64     # conv2 filters
FC  = 128    # fc hidden

# Instantiate model
model = CNN(conv1_filters=C1F, conv2_filters=C2F, fc_hidden=FC)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Model instantiated.")


Model instantiated.


In [ ]:
"""
Load data.
"""

# Download MNIST dataset
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())

# Take subset of 20,000 samples
subset, _ = random_split(full_dataset, [20000, len(full_dataset) - 20000])

# 80/20 train-val split
train_set, val_set = random_split(subset, [16000, 4000])

# Create DataLoaders
train_loader = DataLoader(train_set, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_set):,} | Val: {len(val_set):,}")

100.0%
100.0%
100.0%
100.0%

Train: 16,000 | Val: 4,000


In [5]:
"""
Weights & Biases set up.
"""

# define hyperparameters
config = {
    "learning_rate": LR,
    "batch_size": BS,
    "epochs": EPOCHS,
    "conv1_filters": C1F,
    "conv2_filters": C2F,
    "fc_hidden": FC,
    "optimizer": "adam",
}

# Create wandb run
run = wandb.init(
    entity="dataatuci-wandb-demo",
    project="cnn-mnist",
    name="example-run",                              #TODO: change this to your run name
    config=config,
)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/andrewliu/.netrc.
wandb: Currently logged in as: ahliu7 (ahliu7-uci) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
best_val_loss = float("inf")
best_val_acc  = 0.0
best_model_path = "best_model.pth"

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
        train_correct += (outputs.argmax(1) == labels).sum().item()
    train_loss = train_loss / len(train_loader.dataset)
    train_acc = train_correct / len(train_loader.dataset)

    # Validate
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * len(labels)
            val_correct += (outputs.argmax(1) == labels).sum().item()
    val_loss = val_loss / len(val_loader.dataset)
    val_acc = val_correct / len(val_loader.dataset)

    # Log metrics
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/acc": train_acc,
        "val/loss": val_loss,
        "val/acc": val_acc,
    })

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc  = val_acc
        torch.save(model.state_dict(), best_model_path)

    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}  Best val_acc: {best_val_acc:.4f}")


/Users/andrewliu/Desktop/github/wandb-demo/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/10  train_loss=0.3816  train_acc=0.8846  val_loss=0.1307  val_acc=0.9605
Epoch 2/10  train_loss=0.1025  train_acc=0.9686  val_loss=0.0738  val_acc=0.9758
Epoch 3/10  train_loss=0.0654  train_acc=0.9793  val_loss=0.0856  val_acc=0.9738
Epoch 4/10  train_loss=0.0448  train_acc=0.9852  val_loss=0.0588  val_acc=0.9812
Epoch 5/10  train_loss=0.0350  train_acc=0.9886  val_loss=0.0542  val_acc=0.9828
Epoch 6/10  train_loss=0.0222  train_acc=0.9929  val_loss=0.0505  val_acc=0.9825
Epoch 7/10  train_loss=0.0235  train_acc=0.9914  val_loss=0.0506  val_acc=0.9845
Epoch 8/10  train_loss=0.0124  train_acc=0.9967  val_loss=0.0530  val_acc=0.9838
Epoch 9/10  train_loss=0.0134  train_acc=0.9952  val_loss=0.0612  val_acc=0.9828
Epoch 10/10  train_loss=0.0160  train_acc=0.9943  val_loss=0.0745  val_acc=0.9795

Training complete. Best val_loss: 0.0505  Best val_acc: 0.9825


## 5. Save Best Model as W&B Artifact

Artifacts give every checkpoint a version, making it easy to reproduce any run or deploy a specific model later.


In [7]:
"""
Save best model as W&B artifact.
"""

# Create artifact
artifact = wandb.Artifact(
    name="example-artifact",                        # TODO: change this to your artifact name
    type="model",
    description="Best validation loss checkpoint",
    metadata={"val_loss": best_val_loss, "val_acc": best_val_acc, "epochs": EPOCHS},
)

# Add and log artifact
artifact.add_file(best_model_path)
run.log_artifact(artifact)

print("Artifact saved.")


Artifact saved.


In [8]:
# Make sure to finish the run
wandb.finish()

print(f"Run finished. View it at: {run.url}")


epoch,▁▂▃▃▄▅▆▆▇█
train/acc,▁▆▇▇▇█████
train/loss,█▃▂▂▁▁▁▁▁▁
val/acc,▁▅▅▇▇▇██▇▇
val/loss,█▃▄▂▁▁▁▁▂▃
epoch,10
train/acc,0.99431
train/loss,0.01601
val/acc,0.9795
val/loss,0.07446


Run finished. View it at: https://wandb.ai/dataatuci-wandb-demo/cnn-mnist/runs/bktdose4
